# 工作台仓库： 七层真实配置，以及裸提示词对照

31 节把工作台拆成七层。32–40 节把每一层做成可检查的文件。本节把它们装进**同一个仓库布局**，变成可检出的 harness，而不是聊天窗口里的长提示词。2026 年比拼的不是谁把规则写得更长，而是同一任务上，裸循环和带 harness 的循环谁真的做完。

## 问题描述

团队说「我们已经有工作台了」。打开仓库，只有一段塞进 system prompt 的话：小心一点、记得测试、不要改无关文件。同一模型被丢去「给登陆补输入校验」。它改了四个文件、口头说测试过了、没有状态、没有范围报告、下一会话从零开始。

失败有两种读法。一种怪模型不够强。一种承认：没有磁盘上的七层，完成性无法被判定，只能被叙述。本节补的是后一块：真实配置长什么样，以及用**同一任务、同一验收**对照裸提示词和 harness。

## 基本概念

### 七层在仓库里的落点

层是接口，文件是实现。缺文件的层不算装上。

|层|仓库落点|循环必须做的事|
|---|---|---|
|指令|`AGENTS.md` 路由器 + `docs/agent-rules.md`|启动读路由；规则进 checker，不进百科|
|状态|`agent_state.json` + schema|每轮读写；拒绝瞬时字段|
|范围|`scope_contract.json` + `feature_list.json`|编辑前匹配 glob；一次一个功能|
|反馈|`feedback_record.jsonl`（可轮转）|命令只经运行器；无 exit 不推进|
|验证|`outputs/verification/<task_id>.json`|任务结束读物料；agent 不能自评完成|
|审查|`reviewer_checklist.md` + `review_report.json`|第二循环只读差异，写分不打补丁|
|交接|`clean_state.json` + `handoff.json` / `handoff.md`|先空 blockers，再生成包；下一会话读 JSON|

```mermaid
flowchart LR
A[Bare Prompt] --> B[同一任务]
C[Harness Repo] --> D[七层文件]
D --> E[Binder]
E --> B
B --> F[Completion Scorecard]
```

三文件地板（32）仍是入口。上面六层可以按任务装配，但不能用「提示词里提一句」代替落点。

### 装配器，不是文件堆

把七个文件丢进仓库，再把全文粘进 system prompt，得到的是更胖的裸提示词。harness 是 **binder**：循环在固定时机读固定路径，并把结果当成门，而不是当成散文。

|时机|读什么|门|
|---|---|---|
|会话开始|路由、状态、看板、`init_report.json`|不健康则 halt|
|编辑前|`scope_contract.json`|命中 forbidden / 越界 → block|
|命令后|反馈记录|`exit_code is null` → 拒推进|
|宣称完成前|验证门禁|`passed=false` → 拒 done|
|会话结束前|`clean_state.json`|blockers 非空 → 拒交接|

对照实验里，「带 harness」必须走 binder。只多给一份目录说明、却不接线，记作裸提示词的变体，不记作 harness。

### 对照协议

完成性只能在**锁住变量**之后比。

1. 同一模型、同一温度、同一轮次预算。
2. 同一仓库快照、同一任务（例如：只改 `src/auth/**`，补空密码拒绝，`pytest -q tests/test_login.py` 绿）。
3. 同一外部验收：测试命令、允许/禁止路径、必须留下的状态字段。
4. 两条臂：`bare` 只有任务句子 + 工具；`harness` 同一工具，但 binder 强制七层。
5. 重复 N 次（至少 3），报通过次数，不报单次故事。

裸臂允许模型「自己记得测试」。不算它口头说了，只算磁盘和进程上的事实。harness 臂若跳过 binder，这次 run 作废。

### 完成性记分卡

完成不是「看起来像补丁」。对每次 run 打同一张硬表，分项 0/1，再加总。建议六项，满分 6，**6 才算完成**：

|分项|事实来源|裸臂常见结果|harness 臂要证明|
|---|---|---|---|
|验收命令真跑且 exit 0|反馈 / 进程|空想「测过了」|jsonl 里有该 argv|
|改动 ⊆ 允许 glob，且无 forbidden|最终 diff|顺手改 README / email|scope 报告干净|
|一次只做一个功能|feature_list / 看板|暗主题一起做了|active 只有目标 id|
|block 规则通过|rule_report|无报告|无 block 失败|
|状态可续：有 next_action / 无半成品|agent_state + 工作树|会话结束一句话|字段在、树干净|
|有验证报告（且非自评）|verification_report|模型说 done|门禁 `passed`|

审查分是定性加注，不进「是否完成」。门禁过、审查软失败，仍算完成、但记风险。没有 `next_action` 的结束算未交接，完成性扣分。

对照时输出一张表：臂、N、完成次数、各分项命中率、典型失败。不要输出「harness 更好」这种无分母的句子。

## 什么时候这种模式出错

- **文件在、门不在。** 仓库有七层路径，循环从不读。对照被记成 harness 赢，实际赢的是更长的 prompt。
- **任务太浅。** 「改一个打印」上裸臂也常满分，七层的差被噪声盖住。对照要选会蔓延、会空想测试的任务。
- **验收被换成弱代理。** `echo ok` 当 pytest。两臂都会虚高，harness 的验证层等于没装。
- **用审查分代替完成。** LLM 自评「问题契合 2 分」不能当完成。完成只认记分卡上的磁盘事实。
- **单次叙事。** 一次裸臂翻车、一次 harness 走运，不能当配置结论。没有 N 和分项命中率，就是演示视频。


# 开始编码

对应本章核心：**七层仓库落点**、**binder（时机+门）≠ 文件堆**、**同任务对照 bare / files_only / harness**、**六项记分卡（6 才算完成）**。  
玩具用脚本化两臂保证可断言；生产段用 **LangChain 工具 + DeepSeek** 跑同一张表。不硬凑 PyTorch。


## 1. 教学玩具：WorkbenchRepo + Binder + Scorecard

- `scaffold` 写出七层路径和可失败的登陆验收。
- `bare` 会蔓延且不写反馈；`files_only` 有文件无门；`harness` 必须经 binder。


In [ ]:
from __future__ import annotations

import json
import shutil
import subprocess
import sys
import tempfile
from collections import Counter
from dataclasses import asdict, dataclass, field
from pathlib import Path, PurePosixPath
from typing import Any, Literal


Arm = Literal["bare", "harness", "files_only"]

SCORE_ITEMS = (
    "acceptance_ran",
    "scope_clean",
    "one_feature",
    "rules_block",
    "state_handoff",
    "verification",
)

TASK = "只改 src/auth/**，给登陆补空密码拒绝"


def match_glob(path: str, pattern: str) -> bool:
    """
    Args:
        path: 相对路径。
        pattern: glob。
    Returns:
        matched: 是否命中。
    """
    return PurePosixPath(path.replace("\\", "/")).full_match(pattern.replace("\\", "/"))


@dataclass
class Scorecard:
    """六项 0/1，满分 6 才算完成。"""

    arm: str
    items: dict[str, int]
    notes: list[str] = field(default_factory=list)

    @property
    def total(self) -> int:
        return int(sum(self.items.values()))

    @property
    def complete(self) -> bool:
        return self.total == 6

    def to_dict(self) -> dict[str, Any]:
        return {"arm": self.arm, "items": self.items, "total": self.total, "complete": self.complete, "notes": self.notes}


class BinderError(RuntimeError):
    """binder 门禁拒绝。"""


class WorkbenchRepo:
    """七层真实落点。缺文件的层不算装上。"""

    def __init__(self, root: Path) -> None:
        self.root = Path(root)

    def scaffold(self) -> None:
        """写入七层最小文件 + 可验收的登陆任务。"""
        r = self.root
        (r / "docs").mkdir(parents=True, exist_ok=True)
        (r / "src" / "auth").mkdir(parents=True, exist_ok=True)
        (r / "src" / "email").mkdir(parents=True, exist_ok=True)
        (r / "tests").mkdir(parents=True, exist_ok=True)
        (r / "outputs" / "verification").mkdir(parents=True, exist_ok=True)
        (r / "AGENTS.md").write_text(
            "# router\n- state: agent_state.json\n- rules: docs/agent-rules.md\n- verify: tests/test_login.py\n",
            encoding="utf-8",
        )
        (r / "docs" / "agent-rules.md").write_text(
            "- R-FORB: 不要改 README / src/email\n- R-DOD: 验收命令必须 exit 0\n",
            encoding="utf-8",
        )
        (r / "agent_state.json").write_text(
            json.dumps(
                {
                    "schema_version": 1,
                    "active_task_id": None,
                    "touched_files": [],
                    "assumptions": [],
                    "blockers": [],
                    "next_action": "",
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
        (r / "task_board.json").write_text(
            json.dumps({"stale": False, "tasks": [{"id": "FIX-LOGIN", "status": "todo"}]}, ensure_ascii=False),
            encoding="utf-8",
        )
        (r / "scope_contract.json").write_text(
            json.dumps(
                {
                    "task_id": "T-LOGIN",
                    "goal": TASK,
                    "allowed_files": ["src/auth/**/*.py", "tests/test_login.py"],
                    "forbidden_files": ["README.md", "src/email/**"],
                    "acceptance_criteria": [f"{Path(sys.executable).name} tests/test_login.py"],
                    "rollback_plan": "git checkout -- src/auth",
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
        (r / "feature_list.json").write_text(
            json.dumps(
                {
                    "project": "demo",
                    "active": None,
                    "features": [
                        {"id": "FIX-LOGIN", "status": "todo"},
                        {"id": "DARK-THEME", "status": "todo"},
                    ],
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
        (r / "feedback_record.jsonl").write_text("", encoding="utf-8")
        (r / "src" / "auth" / "login.py").write_text(
            "def login(password: str) -> bool:\n    return True\n",
            encoding="utf-8",
        )
        (r / "src" / "email" / "send.py").write_text("def send():\n    pass\n", encoding="utf-8")
        (r / "README.md").write_text("# demo\n", encoding="utf-8")
        (r / "tests" / "test_login.py").write_text(
            "from pathlib import Path\n"
            "import sys\n"
            "sys.path.insert(0, str(Path(__file__).resolve().parents[1]))\n"
            "from src.auth.login import login\n"
            "assert login('secret') is True\n"
            "assert login('') is False\n"
            "print('ok')\n",
            encoding="utf-8",
        )
        (r / "init_report.json").write_text(json.dumps({"healthy": True}, ensure_ascii=False), encoding="utf-8")
        (r / "reviewer_checklist.md").write_text("# checklist\n五维 0-2\n", encoding="utf-8")

    def layer_files(self) -> dict[str, bool]:
        """
        Returns:
            present: 七层关键路径是否存在。
        """
        return {
            "instruction": (self.root / "AGENTS.md").exists() and (self.root / "docs" / "agent-rules.md").exists(),
            "state": (self.root / "agent_state.json").exists(),
            "scope": (self.root / "scope_contract.json").exists(),
            "feedback": (self.root / "feedback_record.jsonl").exists(),
            "verification": (self.root / "outputs" / "verification").is_dir(),
            "review": (self.root / "reviewer_checklist.md").exists(),
            "handoff": True,  # 生成物，会话结束才有
        }


class Binder:
    """固定时机读固定路径。文件堆进 prompt 不算 harness。"""

    def __init__(self, repo: WorkbenchRepo) -> None:
        self.repo = repo
        self.changed: list[str] = []
        self.started = False

    def start(self) -> None:
        init = json.loads((self.repo.root / "init_report.json").read_text(encoding="utf-8"))
        if not init.get("healthy"):
            raise BinderError("init unhealthy; halt")
        self.started = True
        fl = json.loads((self.repo.root / "feature_list.json").read_text(encoding="utf-8"))
        if fl.get("active") and fl["active"] != "FIX-LOGIN":
            raise BinderError("one feature at a time")
        fl["active"] = "FIX-LOGIN"
        for f in fl["features"]:
            if f["id"] == "FIX-LOGIN":
                f["status"] = "in_progress"
        (self.repo.root / "feature_list.json").write_text(json.dumps(fl, ensure_ascii=False, indent=2), encoding="utf-8")

    def edit(self, rel: str, text: str | None = None) -> None:
        if not self.started:
            raise BinderError("binder.start required")
        contract = json.loads((self.repo.root / "scope_contract.json").read_text(encoding="utf-8"))
        norm = rel.replace("\\", "/").lstrip("./")
        if any(match_glob(norm, g) for g in contract["forbidden_files"]):
            raise BinderError(f"forbidden: {norm}")
        if not any(match_glob(norm, g) for g in contract["allowed_files"]):
            raise BinderError(f"out of scope: {norm}")
        path = self.repo.root / norm
        if text is not None:
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(text, encoding="utf-8")
        if norm not in self.changed:
            self.changed.append(norm)

    def run_acceptance(self) -> dict[str, Any]:
        if not self.started:
            raise BinderError("binder.start required")
        script = self.repo.root / "tests" / "test_login.py"
        proc = subprocess.run([sys.executable, str(script)], capture_output=True, text=True, cwd=self.repo.root)
        row = {
            "command": [Path(sys.executable).name, "tests/test_login.py"],
            "exit_code": proc.returncode,
            "stdout_tail": (proc.stdout or "")[-200:],
        }
        with (self.repo.root / "feedback_record.jsonl").open("a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
        if row["exit_code"] is None:
            raise BinderError("null exit; refuse advance")
        return row

    def write_state(self, *, next_action: str) -> None:
        state = {
            "schema_version": 1,
            "active_task_id": "FIX-LOGIN",
            "touched_files": list(self.changed),
            "assumptions": ["登陆骨架已在仓库"],
            "blockers": [],
            "next_action": next_action,
        }
        (self.repo.root / "agent_state.json").write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")

    def claim_done(self) -> dict[str, Any]:
        feedback = []
        fp = self.repo.root / "feedback_record.jsonl"
        if fp.exists():
            feedback = [json.loads(ln) for ln in fp.read_text(encoding="utf-8").splitlines() if ln.strip()]
        acc = json.loads((self.repo.root / "scope_contract.json").read_text(encoding="utf-8"))["acceptance_criteria"][0]
        ran = any(" ".join(r.get("command", [])) == acc and r.get("exit_code") == 0 for r in feedback)
        scope_ok = True
        contract = json.loads((self.repo.root / "scope_contract.json").read_text(encoding="utf-8"))
        for p in self.changed:
            if any(match_glob(p, g) for g in contract["forbidden_files"]):
                scope_ok = False
        rules_ok = scope_ok and ran
        passed = ran and scope_ok and rules_ok
        report = {"task_id": "T-LOGIN", "passed": passed, "self_declared": False}
        dest = self.repo.root / "outputs" / "verification" / "T-LOGIN.json"
        dest.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
        (self.repo.root / "rule_report.json").write_text(
            json.dumps({"passed": rules_ok, "verdicts": [{"rule_id": "R-DOD-1", "ok": ran, "severity": "block"}]}, ensure_ascii=False),
            encoding="utf-8",
        )
        (self.repo.root / "scope_report.json").write_text(
            json.dumps({"ok": scope_ok, "in_scope": list(self.changed), "forbidden_hits": []}, ensure_ascii=False),
            encoding="utf-8",
        )
        if not passed:
            raise BinderError("verification failed; refuse done")
        return report

    def end_session(self) -> None:
        if (self.repo.root / ".half_applied.diff").exists():
            raise BinderError("dirty tree; refuse handoff")
        clean = {"blockers": []}
        (self.repo.root / "clean_state.json").write_text(json.dumps(clean, ensure_ascii=False), encoding="utf-8")
        state = json.loads((self.repo.root / "agent_state.json").read_text(encoding="utf-8"))
        if not str(state.get("next_action") or "").strip():
            raise BinderError("no next_action; not a handoff")
        pack = {"next_action": state["next_action"], "changed_files": state.get("touched_files", [])}
        (self.repo.root / "handoff.json").write_text(json.dumps(pack, ensure_ascii=False, indent=2), encoding="utf-8")
        (self.repo.root / "handoff.md").write_text(f"# handoff\n{pack['next_action']}\n", encoding="utf-8")
        (self.repo.root / "review_report.json").write_text(
            json.dumps({"verdict": "soft_fail", "total": 6, "role": "reviewer"}, ensure_ascii=False),
            encoding="utf-8",
        )


def apply_login_fix(root: Path) -> None:
    (root / "src" / "auth" / "login.py").write_text(
        "def login(password: str) -> bool:\n    if not password:\n        return False\n    return True\n",
        encoding="utf-8",
    )


def run_bare(repo: WorkbenchRepo, *, creep: bool = True) -> None:
    """裸臂：任务句子 + 直接改文件。可口头宣称完成。"""
    apply_login_fix(repo.root)
    if creep:
        readme = repo.root / "README.md"
        readme.write_text(readme.read_text(encoding="utf-8") + "\n## dark theme coming\n", encoding="utf-8")
        (repo.root / "src" / "email" / "send.py").write_text("def send():\n    return 'retry'\n", encoding="utf-8")
    # 空想测试：不写 feedback，不写 verification


def run_files_only(repo: WorkbenchRepo) -> None:
    """文件在、门不在：七层都在磁盘，但不走 binder。"""
    run_bare(repo, creep=True)


def run_harness(repo: WorkbenchRepo) -> None:
    """harness 臂：同一任务，必须经 binder。"""
    b = Binder(repo)
    b.start()
    apply_login_fix(repo.root)
    b.edit("src/auth/login.py")
    row = b.run_acceptance()
    if row["exit_code"] != 0:
        raise BinderError("acceptance red")
    b.write_state(next_action="等审查后合并 FIX-LOGIN")
    b.claim_done()
    b.end_session()


def score(repo: WorkbenchRepo, arm: str) -> Scorecard:
    """
    Args:
        repo: 跑完后的仓库。
        arm: 臂名。
    Returns:
        card: 六项记分卡。
    """
    items = {k: 0 for k in SCORE_ITEMS}
    notes: list[str] = []
    acc = json.loads((repo.root / "scope_contract.json").read_text(encoding="utf-8"))["acceptance_criteria"][0]
    feedback = []
    fp = repo.root / "feedback_record.jsonl"
    if fp.exists() and fp.stat().st_size:
        feedback = [json.loads(ln) for ln in fp.read_text(encoding="utf-8").splitlines() if ln.strip()]
    if any(" ".join(r.get("command", [])) == acc and r.get("exit_code") == 0 for r in feedback):
        items["acceptance_ran"] = 1
    else:
        notes.append("no real acceptance run")

    contract = json.loads((repo.root / "scope_contract.json").read_text(encoding="utf-8"))
    changed: list[str] = []
    # 用内容探测实际改动
    if "dark theme" in (repo.root / "README.md").read_text(encoding="utf-8"):
        changed.append("README.md")
    send = (repo.root / "src" / "email" / "send.py").read_text(encoding="utf-8")
    if "retry" in send:
        changed.append("src/email/send.py")
    login = (repo.root / "src" / "auth" / "login.py").read_text(encoding="utf-8")
    if "not password" in login:
        changed.append("src/auth/login.py")
    state_p = repo.root / "agent_state.json"
    if state_p.exists():
        st = json.loads(state_p.read_text(encoding="utf-8"))
        for p in st.get("touched_files") or []:
            if p not in changed:
                changed.append(p)
    forbidden = [p for p in changed if any(match_glob(p, g) for g in contract["forbidden_files"])]
    allowed_ok = all(
        any(match_glob(p, g) for g in contract["allowed_files"]) for p in changed if p not in forbidden
    )
    if changed and not forbidden and allowed_ok:
        items["scope_clean"] = 1
    else:
        notes.append(f"scope creep={forbidden or changed}")

    fl = json.loads((repo.root / "feature_list.json").read_text(encoding="utf-8"))
    if fl.get("active") == "FIX-LOGIN" and not any(f["id"] == "DARK-THEME" and f["status"] == "in_progress" for f in fl["features"]):
        items["one_feature"] = 1
    else:
        notes.append(f"features active={fl.get('active')}")

    rp = repo.root / "rule_report.json"
    if rp.exists() and json.loads(rp.read_text(encoding="utf-8")).get("passed"):
        items["rules_block"] = 1
    else:
        notes.append("no rule_report or block fail")

    st = json.loads((repo.root / "agent_state.json").read_text(encoding="utf-8")) if state_p.exists() else {}
    dirty = (repo.root / ".half_applied.diff").exists()
    if str(st.get("next_action") or "").strip() and not dirty:
        items["state_handoff"] = 1
    else:
        notes.append("state not handoff-ready")

    vp = repo.root / "outputs" / "verification" / "T-LOGIN.json"
    if vp.exists():
        v = json.loads(vp.read_text(encoding="utf-8"))
        if v.get("passed") and not v.get("self_declared"):
            items["verification"] = 1
        else:
            notes.append("gate not passed")
    else:
        notes.append("no verification report")

    return Scorecard(arm=arm, items=items, notes=notes)


def compare_arms(*, n: int = 3) -> dict[str, Any]:
    """
    锁变量对照：同任务重复 n 次。

    Args:
        n: 重复次数。
    Returns:
        table: 各臂完成次数与分项命中率。
    """
    arms: dict[str, list[Scorecard]] = {"bare": [], "files_only": [], "harness": []}
    for i in range(n):
        for arm, runner in (
            ("bare", lambda r: run_bare(r, creep=True)),
            ("files_only", run_files_only),
            ("harness", run_harness),
        ):
            tmp = Path(tempfile.mkdtemp(prefix=f"wb_{arm}_{i}_"))
            try:
                repo = WorkbenchRepo(tmp)
                repo.scaffold()
                runner(repo)
                arms[arm].append(score(repo, arm))
            finally:
                shutil.rmtree(tmp, ignore_errors=True)

    def summarize(name: str, cards: list[Scorecard]) -> dict[str, Any]:
        hits = Counter()
        for c in cards:
            for k, v in c.items.items():
                hits[k] += v
        return {
            "arm": name,
            "N": len(cards),
            "complete": sum(1 for c in cards if c.complete),
            "item_hit_rate": {k: round(hits[k] / len(cards), 2) for k in SCORE_ITEMS},
            "typical_notes": cards[0].notes if cards else [],
        }

    return {k: summarize(k, v) for k, v in arms.items()}


print("workbench repo ready | 7-layer scaffold + binder + scorecard")


## 2. 玩具示例

七层在场、裸臂不完成、binder 拒 forbidden、N=3 对照表。


In [ ]:
def demo_workbench_repo() -> None:
    """七层落点、裸臂失败、files_only 不等于 harness、对照表。"""
    root = Path(tempfile.mkdtemp(prefix="wb_layers_"))
    try:
        repo = WorkbenchRepo(root)
        repo.scaffold()
        layers = repo.layer_files()
        assert all(layers[k] for k in layers if k != "handoff")
        print("seven layer files present ok")

        run_bare(repo, creep=True)
        bare = score(repo, "bare")
        assert not bare.complete
        assert bare.items["acceptance_ran"] == 0
        assert bare.items["scope_clean"] == 0
        print("bare incomplete ok", bare.to_dict())
    finally:
        shutil.rmtree(root, ignore_errors=True)

    hroot = Path(tempfile.mkdtemp(prefix="wb_h_"))
    try:
        h = WorkbenchRepo(hroot)
        h.scaffold()
        run_harness(h)
        card = score(h, "harness")
        assert card.complete and card.total == 6
        assert (hroot / "handoff.json").exists()
        print("harness complete ok")

        b = Binder(h)
        b.started = True
        try:
            b.edit("README.md", "nope")
            raise AssertionError("forbidden")
        except BinderError:
            print("binder blocks forbidden ok")
    finally:
        shutil.rmtree(hroot, ignore_errors=True)

    table = compare_arms(n=3)
    assert table["bare"]["complete"] == 0
    assert table["files_only"]["complete"] == 0
    assert table["harness"]["complete"] == 3
    print("=== compare N=3 ===")
    print(json.dumps(table, ensure_ascii=False, indent=2))
    print("TOY DEMO OK")


demo_workbench_repo()


## 3. 生产级：LangChain 对照工具 + DeepSeek

工具：`run_bare_arm` / `run_files_only_arm` / `run_harness_arm` / `compare_scorecard`。需 `DEEPSEEK_API_KEY`。


In [ ]:
import json
import os
import shutil
import sys
import tempfile
import threading
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"
PROD_ROOT: Path | None = None
PROD_REPO: WorkbenchRepo | None = None
PROD_BINDER: Binder | None = None
LAST_CARDS: list[dict[str, Any]] = []
PROD_LOCK = threading.Lock()


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def reset_compare_repo() -> str:
    """
    Returns:
        json: 新快照路径。
    """
    global PROD_ROOT, PROD_REPO, PROD_BINDER
    if PROD_ROOT and PROD_ROOT.exists():
        shutil.rmtree(PROD_ROOT, ignore_errors=True)
    PROD_ROOT = Path(tempfile.mkdtemp(prefix="wb_prod_"))
    PROD_REPO = WorkbenchRepo(PROD_ROOT)
    PROD_REPO.scaffold()
    PROD_BINDER = Binder(PROD_REPO)
    return json.dumps({"root": str(PROD_ROOT), "task": TASK}, ensure_ascii=False)


def _repo() -> WorkbenchRepo:
    if PROD_REPO is None:
        reset_compare_repo()
    assert PROD_REPO is not None
    return PROD_REPO


class EmptyArgs(BaseModel):
    pass


class CreepArgs(BaseModel):
    creep: bool = True


def build_compare_tools() -> list[StructuredTool]:
    def _reset(**kwargs: Any) -> str:
        with PROD_LOCK:
            LAST_CARDS.clear()
            return reset_compare_repo()

    def _layers(**kwargs: Any) -> str:
        with PROD_LOCK:
            return json.dumps(_repo().layer_files(), ensure_ascii=False)

    def _bare(**kwargs: Any) -> str:
        with PROD_LOCK:
            reset_compare_repo()
            run_bare(_repo(), creep=CreepArgs(**kwargs).creep)
            card = score(_repo(), "bare")
            LAST_CARDS.append(card.to_dict())
            return json.dumps(card.to_dict(), ensure_ascii=False)

    def _files_only(**kwargs: Any) -> str:
        with PROD_LOCK:
            reset_compare_repo()
            run_files_only(_repo())
            card = score(_repo(), "files_only")
            LAST_CARDS.append(card.to_dict())
            return json.dumps(card.to_dict(), ensure_ascii=False)

    def _harness(**kwargs: Any) -> str:
        with PROD_LOCK:
            reset_compare_repo()
            run_harness(_repo())
            card = score(_repo(), "harness")
            LAST_CARDS.append(card.to_dict())
            return json.dumps(card.to_dict(), ensure_ascii=False)

    def _table(**kwargs: Any) -> str:
        with PROD_LOCK:
            table = compare_arms(n=3)
            return json.dumps(table, ensure_ascii=False)

    def _try_forbidden(**kwargs: Any) -> str:
        with PROD_LOCK:
            reset_compare_repo()
            b = Binder(_repo())
            b.start()
            try:
                b.edit("README.md", "x")
                return json.dumps({"ok": True}, ensure_ascii=False)
            except BinderError as e:
                return json.dumps({"ok": False, "error": str(e)}, ensure_ascii=False)

    return [
        StructuredTool.from_function(name="reset_repo", description="重建七层快照。", func=_reset, args_schema=EmptyArgs),
        StructuredTool.from_function(name="list_layers", description="七层文件是否在。", func=_layers, args_schema=EmptyArgs),
        StructuredTool.from_function(name="run_bare_arm", description="裸提示词臂：直接改文件。", func=_bare, args_schema=CreepArgs),
        StructuredTool.from_function(
            name="run_files_only_arm",
            description="文件在、门不在：不算 harness。",
            func=_files_only,
            args_schema=EmptyArgs,
        ),
        StructuredTool.from_function(name="run_harness_arm", description="经 binder 的 harness 臂。", func=_harness, args_schema=EmptyArgs),
        StructuredTool.from_function(name="compare_scorecard", description="N=3 对照表：完成次数+分项命中率。", func=_table, args_schema=EmptyArgs),
        StructuredTool.from_function(name="try_forbidden_edit", description="演示 binder 拒 README。", func=_try_forbidden, args_schema=EmptyArgs),
    ]


COMPARE_TOOLS = build_compare_tools()


def build_compare_agent():
    """
    Returns:
        agent: 跑对照、只报记分卡数字。
    """
    system = (
        "你是工作台对照代理。同一任务上跑 bare / files_only / harness，"
        "再 compare_scorecard。不要说「harness 更好」除非带上 N 和分项命中率。\n"
        "files_only 不是 harness。用中文解释记分卡。"
    )
    return create_agent(get_llm(), COMPARE_TOOLS, system_prompt=system)


def _print_trace(messages: list[BaseMessage], limit: int = 28) -> None:
    n = 0
    for m in messages:
        if n >= limit:
            break
        if isinstance(m, HumanMessage):
            print(f"USER: {m.content}")
            n += 1
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    print(f"ACTION: {tc['name']}({tc.get('args', {})})")
                    n += 1
            elif m.content:
                print(f"ASSISTANT: {str(m.content)[:320]}")
                n += 1
        elif isinstance(m, ToolMessage):
            print(f"OBS[{m.name}]: {str(m.content)[:240]}")
            n += 1


print(f"workbench repo production ready | {MODEL}")


## 4. 生产示例

无 `DEEPSEEK_API_KEY` 则跳过 LLM；脚本路径仍输出 N=3 分项命中率。


In [ ]:
def demo_prod_compare() -> None:
    """脚本对照表 +（有 key 时）DeepSeek。"""
    table = compare_arms(n=3)
    assert table["bare"]["complete"] == 0
    assert table["files_only"]["complete"] == 0
    assert table["harness"]["complete"] == 3
    print("=== scripted N=3 ===")
    print(json.dumps(table, ensure_ascii=False, indent=2))

    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP llm agent: DEEPSEEK_API_KEY missing")
        print("PROD DEMO OK (scripted only)")
        return

    agent = build_compare_agent()
    prompt = (
        "reset_repo，list_layers，run_bare_arm，run_files_only_arm，run_harness_arm，"
        "try_forbidden_edit，compare_scorecard，"
        "用中文按表解释：为什么 files_only 完成次数仍是 0。"
    )
    result = agent.invoke({"messages": [HumanMessage(content=prompt)]})
    print("=== control agent ===")
    _print_trace(list(result["messages"]))
    print("PROD DEMO OK")


demo_prod_compare()
